# Portofolio Analisis Data: Video Game Sales

Notebook ini berisi analisis data penjualan video game global berdasarkan dataset `vgsales.csv`. Analisis ini terbagi menjadi dua fokus utama:
1. **Menganalisis Tren Kesuksesan Platform/Konsol**: Menyelidiki siklus hidup dan dominasi platform video game dari masa ke masa.
2. **Strategi Monopoli Publisher**: Membedah strategi bisnis dari 5 publisher game tersukses dan penguasaan genre market mereka.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Memuat dataset
df = pd.read_csv('vgsales.csv')

# Menampilkan informasi awal dataset
display(df.head())
display(df.info())

## Pra-pemrosesan Data
Sebelum melakukan analisis, kita perlu memastikan bahwa dataset bersih. Langkah utamanya adalah menghapus nilai kosong (*missing values*) pada kolom esensial seperti `Year` dan `Publisher`, serta menyesuaikan tipe data tahun rilis menjadi format bilangan bulat (integer).

In [ ]:
# Menghapus baris yang tidak memiliki informasi Tahun (Year) atau Publisher
df_clean = df.dropna(subset=['Year', 'Publisher']).copy()

# Mengubah tipe data Year menjadi integer
df_clean['Year'] = df_clean['Year'].astype(int)

print(f"Jumlah baris sebelum pra-pemrosesan : {len(df)}")
print(f"Jumlah baris setelah pra-pemrosesan  : {len(df_clean)}")

## 1. Menganalisis Tren Kesuksesan Platform/Konsol (Fokus Sejarah & Tren)
**Topik:** Siklus hidup dan dominasi konsol video game dari masa ke masa.

**Pertanyaan Analisis:** "Platform/konsol apa yang memiliki umur pasar paling panjang dan menyumbang total penjualan global tertinggi sepanjang sejarah?"

**Langkah-langkah Analisis Deskriptif:**
- Menghitung rentang tahun rilis game untuk setiap platform (Lifespan/Umur Pasar).
- Menghitung total penjualan global (Total Global Sales) masing-masing platform.
- Menggabungkan visualisasi kedua atribut tersebut untuk memberikan insight yang komprehensif.

In [ ]:
# 1. Menghitung lifespan (umur pasar) platform
platform_years = df_clean.groupby('Platform')['Year'].agg(
    Tahun_Rilis_Pertama='min', 
    Tahun_Rilis_Terakhir='max'
)
# Umur pasar dihitung dari selisih tahun rilis terakhir dan pertama (ditambah 1 agar tahun rilis saja tetap dihitung minimal 1 tahun umur)
platform_years['Umur_Pasar_(Tahun)'] = platform_years['Tahun_Rilis_Terakhir'] - platform_years['Tahun_Rilis_Pertama'] + 1

# 2. Menghitung total penjualan global platform
platform_sales = df_clean.groupby('Platform')['Global_Sales'].sum().reset_index()
platform_sales.rename(columns={'Global_Sales': 'Total_Penjualan_Global'}, inplace=True)

# Menggabungkan data
platform_analysis = pd.merge(platform_sales, platform_years, on='Platform')

# Mengambil 10 platform pendominasi (berdasarkan total sales terbesar)
top_platforms = platform_analysis.sort_values(by='Total_Penjualan_Global', ascending=False).head(10)
display(top_platforms)

In [ ]:
# Visualisasi Total Penjualan Global dan Umur Pasar dalam satu chart
fig, ax1 = plt.subplots(figsize=(14, 7))

# Bar chart untuk Total Penjualan Global
sns.barplot(x='Platform', y='Total_Penjualan_Global', data=top_platforms, color='skyblue', ax=ax1, label='Total Penjualan Global (Juta Kopi)')
ax1.set_ylabel('Total Penjualan Global (Juta Kopi)', fontsize=12, color='blue')
ax1.set_xlabel('Platform/Konsol', fontsize=12)
ax1.set_title('Top 10 Platform Sepanjang Sejarah: Penjualan Global vs Umur Pasar', fontsize=16, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='blue')

# Line chart untuk Umur Pasar
ax2 = ax1.twinx()
sns.lineplot(x='Platform', y='Umur_Pasar_(Tahun)', data=top_platforms, color='red', marker='o', ax=ax2, label='Umur Pasar (Tahun)', linewidth=2.5)
ax2.set_ylabel('Umur Pasar (Tahun)', fontsize=12, color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.grid(False)

# Menyesuaikan posisi legenda agar tidak tumpang tindih
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper left')

plt.show()

### Kesimpulan Analisis (Tren Platform)
Berdasarkan visualisasi di atas:
- **PS2** adalah rajanya konsol di sepanjang masa. Ia membukukan total penjualan *(Global Sales)* tertinggi yakni lebih dari 1.200 juta kopi. Selain memegang takhta penjualan, umur pasarnya terhitung sangat mapan dan produktif (mencapai 12 tahun umur rentang perilisan game).
- Generasi konsol seperti **X360, PS3, dan Wii** menempati persaingan yang amat ketat di baliknya dengan umur pasar di kisaran rata-rata 10-11 tahun, didampingi konsol tua **PS** dan *handheld* legendaris **DS**.
- **PC (Personal Computer)** terbukti unik: meskipun rekam jejak penjualannya (terutama rilis versi fisik pada masanya) kalah tinggi dibanding konsol-konsol *gaming* terdedikasi di era keemasannya, PC mewakili pasar yang paling tahan banting dan bersiklus hidup paling panjang (32 tahun dalam rentang rilis pada dataset). Hal ini karena ekosistem PC berevolusi secara modular dan sangat berbeda dari siklus *generasional* yang kaku seperti pada konsol rumah.

## 2. Strategi Monopoli Publisher (Fokus Bisnis)
**Topik:** Strategi bisnis dari perusahaan pembuat game (Publisher) papan atas.

**Pertanyaan Analisis:** "Siapakah 5 Publisher paling sukses berdasarkan penjualan global, dan Genre game spesifik apa yang menjadi penyumbang pendapatan terbesar bagi masing-masing publisher tersebut?"

**Langkah-langkah Analisis Deskriptif & Preskriptif:**
- Menemukan 5 (Top 5) Publisher berdasarkan Total Penjualan Global.
- Mengelompokkan penjualan kelima publisher ini berdasarkan Genre masing-masing.
- Merumuskan *insight* taktis berdasar spesialisasi penerbit di genre terkait.

In [ ]:
# 1. Mengidentifikasi Top 5 Publisher
top5_publishers = df_clean.groupby('Publisher')['Global_Sales'].sum().nlargest(5)
top5_publisher_names = top5_publishers.index.tolist()

print("=== TOP 5 PUBLISHER GLOBAL ===")
for i, (pub, sales) in enumerate(top5_publishers.items(), 1):
    print(f"{i}. {pub} ({sales:.2f} Juta Kopi)")

# 2. Memfilter dan mengambil data hanya untuk kelima publisher elit ini
df_top5_pub = df_clean[df_clean['Publisher'].isin(top5_publisher_names)]

# Menghitung sumbangsih masing-masing Genre untuk kelima Publisher
pub_genre_sales = df_top5_pub.groupby(['Publisher', 'Genre'])['Global_Sales'].sum().reset_index()

# Menemukan Genre andalan (The Golden Genre) untuk masing-masing Publisher
winning_genres = []
for pub in top5_publisher_names:
    pub_data = pub_genre_sales[pub_genre_sales['Publisher'] == pub]
    top_genre = pub_data.sort_values(by='Global_Sales', ascending=False).iloc[0]
    
    winning_genres.append({
        'Publisher': pub, 
        'Golden Genre': top_genre['Genre'], 
        'Genre Sales (Jt Kopi)': top_genre['Global_Sales'],
        'Total Pub Sales (Jt Kopi)': top5_publishers[pub],
        '% Dominasi Genre': (top_genre['Global_Sales'] / top5_publishers[pub]) * 100
    })

display(pd.DataFrame(winning_genres).round(2).set_index('Publisher'))

In [ ]:
# Visualisasi Distribusi Penjualan Genre oleh Top 5 Publisher
plt.figure(figsize=(16, 8))

# Menampilkan grafik batang ganda yang dipisah oleh Hue (Genre)
sns.barplot(x='Publisher', y='Global_Sales', hue='Genre', data=pub_genre_sales, 
            order=top5_publisher_names, palette='Paired')

plt.title('Strategi Monopoli Market: Penguasaan Genre oleh 5 Publisher Teratas', fontsize=16, fontweight='bold')
plt.ylabel('Total Penjualan Global (Juta Kopi)', fontsize=12)
plt.xlabel('Nama Publisher', fontsize=12)
plt.legend(title='Genre Market', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Kesimpulan Analisis Deskriptif & Saran Preskriptif (Strategi Bisnis Publisher)

**Analisis Deskriptif:**
Setiap raksasa industri game *(Top 5 Publisher)* memiliki **ladang uang (Golden Genre)** dan keahlian spesifiknya berskala monopoli:
1. **Nintendo**, pemuncak sejarah (lebih dari 1.784 juta kopi terjual), menancapkan dominasi mutlak di **Platform** (didongkrak IP fenomenal Super Mario) yang meraup porsi lebih dari 23% pendapatannya sendiri, diiringi ketangguhan merata di *Role-Playing* (Pokemon) dan *Sports* (Wii Sports).
2. **Electronic Arts (EA)** merajai mutlak di ekosistem **Sports** (berkat megatrend semacam *FIFA* dan *Madden NFL*). Monopolinya di genre olahraga secara konsisten sukses mencetak ±43,8% pendapatan total keseluruhan grupnya.
3. **Activision** merupakan diktator di jagad game **Shooter**. Hanya dari game tembak-menembak saja (ditarik kencang layaknya kuda oleh satu kekuatan masif `Call of Duty`), ia mendulang nyaris **41,4%** omset bisnis rekam jejak mereka.
4. **Sony Computer Entertainment** memimpin area **Racing** (dipelopori *Gran Turismo*) dan bervariasi amat tangguh di ranah Action serta Platform.
5. **Take-Two Interactive** mengorbit pada pilar raksasa utamanya di genre **Action** (paling utama memerah kekuatan tanpa tanding *Grand Theft Auto*); genre Action mengeruk *revenue* dalam porsi menakjubkan, yaitu nyaris **57,7%** pendapatan sejauh ini bagi Take-Two.

**Analisis Preskriptif (Rekomendasi Strategi / *Actionable Insights* Bagi Publisher/Developer Lain):**
- **Hindari Medan Tempur Utama ("Red Ocean"):** Sangat tidak logis dan berisiko fatal jika sebuah studio menengah atau publisher pendatang memaksakan diri bersaing secara berhadapan-hadapan (*head-to-head*) pada porsi utama para penguasa ini. Mencoba meluncurkan game simulasi sepakbola konvensional untuk menggeser **EA (Sports)**, membakar ratusan juta Dollar AS guna membuat game militer sinematik guna menantang rajanya FPS **Activision (Shooter)**, atau membuat game lompat jangkit (maskot *platforming*) demi menumbangkan **Nintendo** adalah langkah "bunuh diri" finansial. Ekosistem ini dibentengi kuat oleh monopoli, dominasi teknologi, fanatisme *loyalty* pemainnya.
- **Invasi di Lahan Luang ("Blue Ocean"):** Publisher / studio dianjurkan meramu proyeksi untuk menggarap pasar di genre yang **skalanya besar, masih tersebar lebih luas, dan belum dikunci mutlak (belum monopolistik)**. Misalnya terjun merengkuh pecinta *Adventure, Puzzle, atau Strategy*. 
- **Strategi "*Hybrid / Sub-Genre*":** Langkah taktis lainnya adalah membangun *Genre Hybrid*, menyatukan *Shooter* dan *Role-Playing* (seperti *Destiny / Borderlands* yang sukses curi pasar), atau menggabung *Action* dengan *Simulation*. Pendekatan semacam ini bisa menciptakan kategori dan segmentasi gamer independen (*niche*) yang tidak bertabrakan dengan kepentingan dan dominasi absolut kelompok di atas.